In [ ]:
import os
from glob import glob
import geopandas
import pandas
import fiona
import numpy
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import rasterio
# from analysis_utils import *

In [ ]:
import geopandas as gpd
import rasterio
from rasterio.mask import mask
import numpy as np
from shapely.geometry import Point
import pandas as pd
import cartopy.crs as ccrs
import rioxarray
from scipy.spatial import cKDTree


In [ ]:
# Load DEM file and landslides CSV as GeoDataFrame
dem_jamaica = 'L:\\jamaica\\DEM_30m.tif'
landslides = 'L:\\jamaica\\JamaicaLandslides.csv'
dem_jam = rasterio.open(dem_jamaica)
landslides_df = pd.read_csv(landslides)

In [ ]:
#Check crs of dem
with rasterio.open(dem_jamaica) as src:
    crs = src.crs
print(crs)

In [ ]:
#Reproject dem to crs epsg: 3448
dem_file = rioxarray.open_rasterio(dem_jamaica)
crs_correct = 'EPSG:3448'
dem_jam_3448 = dem_file.rio.reproject(crs_correct)


#dem_jam_3448 = dem_jamaica.to_crs(epsg=crs_correct)

#data = data.to_crs(epsg=jamaica_crs)

In [ ]:
#Check crs of dem_3448
crs_3448 = dem_jam_3448.rio.crs
print(crs_3448)

In [ ]:
#Plot the DEM file and landslide points
fig, ax = plt.subplots(figsize = 10, 14)


In [ ]:
landslides_df

In [ ]:
# Buffer radius in meters
buffer_radius = 50

# Function to calculate slope from a DEM array
def calculate_slope(dem_array, transform):
    slope_x, slope_y = np.gradient(dem_array, transform[0], transform[4])
    slope = np.arctan(np.sqrt(slope_x ** 2 + slope_y ** 2)) * (180.0 / np.pi)
    return slope

# Prepare empty results list
results = []

# Iterate over each landslide point
for index, landslide in landslides_df.iterrows():
    lon, lat = landslide['longitude'], landslide['latitude']
    if np.isnan(lon) or np.isnan(lat):
        continue

    # Create Point geometry
    point = Point(lon, lat)
    
    # Buffer the point
    buffer_geometry = point.buffer(buffer_radius)
    
    # Mask DEM within buffer
    try:
        masked, _ = mask(dem_jam, shapes=[buffer_geometry], crop=True)
    except ValueError:
        print(f"Error masking DEM for index {index}")
        continue

    # Calculate slope within buffer
    slope_array = calculate_slope(masked[0], dem_jam.transform)
    
    # Find steepest slope
    max_slope_index = np.unravel_index(np.argmax(slope_array), slope_array.shape)
    max_slope_latlon = rasterio.transform.xy(dem_jam.transform, max_slope_index[0], max_slope_index[1])
    max_slope = slope_array[max_slope_index]
    
    # Store results
    result = {
        'lat': max_slope_latlon[1],
        'lon': max_slope_latlon[0],
        'slope': max_slope,
        'date': landslide['ev_date'],
        'Name': landslide['div_name']
    }
    results.append(result)

# Create a new GeoDataFrame with the results
steepest_slopes_gdf = gpd.GeoDataFrame(results, geometry=gpd.points_from_xy([result['lon'] for result in results],
                                                                          [result['lat'] for result in results]))

# Print or save the GeoDataFrame
print(steepest_slopes_gdf)

In [ ]:
#Using KDTree to obatin nearest steepest slope
geometry = [Point(xy) for xy in zip(landslides_df['longitude'], landslides_df['latitude'])]
gdf = gpd.GeoDataFrame(landslides_df, geometry = geometry)
coordinates = np.array(list(gdf.geometry.apply(lambda point: (point.x, point.y))))

tree = cKDTree(coordinates)

def calculate_slope(dem_array, cell_size):
    # Calculate slope using np.gradient
    dz_dx, dz_dy = np.gradient(dem_array, cell_size, cell_size)
    # Calculate slope magnitude
    slope_rad = np.arctan(np.sqrt(dz_dx**2 + dz_dy**2))
    # Convert slope to degrees
    slope = np.degrees(slope_rad)
    return slope

gdf['steepestSlope'] = None

for idx, row in gdf.iterrows():
    _, indices = tree.query((row.geometry.x, row.geometry.y), k=10, distance_upper_bound=100)
    slopes = [calculate_slope(tree.data[i], 5) for i in indices if i != len(tree.data)]
    if slopes:
        gdf.at[idx, 'steepestSlope'] = max(slopes)
        
print(gdf)


In [ ]:
steepest_slopes_gdf.to_csv("L:\\jamaica\\steepest_slope_gdf.csv")